# 10 — BBBC047 / ADMET Overlap Check

Check overlap between the BBBC047 pre-training dataset (109K compounds) and the ADMET benchmark test sets (22 tasks). If overlap exists, create a filtered version with ADMET test SMILES removed.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem

BBBC047_PATH = Path("/home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings.csv")
ADMET_TEST_PATH = Path("/home/shpark/prj-molrepr/datacache/admet_smiles/admet_test_smiles.json")
ADMET_ALL_PATH = Path("/home/shpark/prj-molrepr/datacache/admet_smiles/admet_all_smiles.json")
ADMET_PER_TASK_PATH = Path("/home/shpark/prj-molrepr/datacache/admet_smiles/admet_smiles_per_task.json")

def canonicalize(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return smi
    return Chem.MolToSmiles(mol)

# Load BBBC047
bbbc_df = pd.read_csv(BBBC047_PATH, usecols=["SMILES"])
bbbc_df["SMILES_canon"] = bbbc_df["SMILES"].apply(canonicalize)
bbbc_smiles = set(bbbc_df["SMILES_canon"].tolist())

# Load ADMET SMILES
with open(ADMET_TEST_PATH) as f:
    admet_test = set(json.load(f))
with open(ADMET_ALL_PATH) as f:
    admet_all = set(json.load(f))
with open(ADMET_PER_TASK_PATH) as f:
    admet_per_task = json.load(f)

print(f"BBBC047 unique SMILES:   {len(bbbc_smiles):,}")
print(f"ADMET test SMILES:       {len(admet_test):,}")
print(f"ADMET all SMILES:        {len(admet_all):,}")

BBBC047 unique SMILES:   109,645
ADMET test SMILES:       13,818
ADMET all SMILES:        46,202


## Overall overlap

In [2]:
overlap_test = bbbc_smiles & admet_test
overlap_all = bbbc_smiles & admet_all

print(f"=== BBBC047 ↔ ADMET Overlap ===")
print(f"  BBBC047 ∩ ADMET test:  {len(overlap_test):,}  ({len(overlap_test)/len(bbbc_smiles)*100:.2f}% of BBBC047)")
print(f"  BBBC047 ∩ ADMET all:   {len(overlap_all):,}  ({len(overlap_all)/len(bbbc_smiles)*100:.2f}% of BBBC047)")
print(f"  ADMET test coverage:   {len(overlap_test)/len(admet_test)*100:.1f}% of ADMET test found in BBBC047")
print(f"  ADMET all coverage:    {len(overlap_all)/len(admet_all)*100:.1f}% of ADMET all found in BBBC047")

=== BBBC047 ↔ ADMET Overlap ===
  BBBC047 ∩ ADMET test:  974  (0.89% of BBBC047)
  BBBC047 ∩ ADMET all:   2,429  (2.22% of BBBC047)
  ADMET test coverage:   7.0% of ADMET test found in BBBC047
  ADMET all coverage:    5.3% of ADMET all found in BBBC047


## Per-task overlap breakdown

In [3]:
print(f"{'Task':<40} {'Test size':>10} {'Overlap':>10} {'%':>8}")
print("-" * 70)

task_overlaps = {}
for task, smi_dict in admet_per_task.items():
    test_smi = set(smi_dict["test"])
    overlap = bbbc_smiles & test_smi
    task_overlaps[task] = {"test_size": len(test_smi), "overlap": len(overlap)}
    pct = len(overlap) / len(test_smi) * 100 if test_smi else 0
    marker = " <<<" if len(overlap) > 0 else ""
    print(f"  {task:<38} {len(test_smi):>10,} {len(overlap):>10,} {pct:>7.1f}%{marker}")

total_overlap = sum(v["overlap"] for v in task_overlaps.values())
print(f"\n  {'TOTAL (sum across tasks)':<38} {sum(v['test_size'] for v in task_overlaps.values()):>10,} {total_overlap:>10,}")
print(f"  {'UNIQUE test SMILES in overlap':<38} {len(overlap_test):>10,}")

Task                                      Test size    Overlap        %
----------------------------------------------------------------------
  caco2_wang                                    181         21    11.6% <<<
  hia_hou                                       117         24    20.5% <<<
  pgp_broccatelli                               245         47    19.2% <<<
  bioavailability_ma                            128         37    28.9% <<<
  lipophilicity_astrazeneca                     840         48     5.7% <<<
  solubility_aqsoldb                          1,997        173     8.7% <<<
  bbb_martins                                   394         62    15.7% <<<
  ppbr_az                                       343         34     9.9% <<<
  vdss_lombardo                                 221         30    13.6% <<<
  cyp2d6_veith                                2,626        157     6.0% <<<
  cyp3a4_veith                                2,467        122     4.9% <<<
  cyp2c9_veith       

## Create filtered BBBC047 dataset (remove ADMET test SMILES)

In [4]:
# Load full BBBC047 dataset (all columns)
bbbc_full = pd.read_csv(BBBC047_PATH)
bbbc_full["SMILES_canon"] = bbbc_full["SMILES"].apply(canonicalize)

# Filter
mask_keep = ~bbbc_full["SMILES_canon"].isin(admet_test)
bbbc_filtered = bbbc_full[mask_keep].drop(columns=["SMILES_canon"]).reset_index(drop=True)

print(f"BBBC047 filtering:")
print(f"  Before:  {len(bbbc_full):,} compounds")
print(f"  Removed: {len(bbbc_full) - len(bbbc_filtered):,} (ADMET test overlap)")
print(f"  After:   {len(bbbc_filtered):,} compounds")

# Save
OUTPUT_PATH = Path("/home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings_filtered.csv")
bbbc_filtered.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1e6:.1f} MB")

# Verify
verify = pd.read_csv(OUTPUT_PATH, usecols=["SMILES"])
verify_canon = set(verify["SMILES"].apply(canonicalize).tolist())
leak = verify_canon & admet_test
print(f"\nVerification: {len(leak)} ADMET test SMILES remaining (should be 0)")
assert len(leak) == 0, f"Leakage: {len(leak)}"
print("PASS: No data leakage")

BBBC047 filtering:
  Before:  109,645 compounds
  Removed: 974 (ADMET test overlap)
  After:   108,671 compounds



Saved to: /home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings_filtered.csv
File size: 744.8 MB



Verification: 0 ADMET test SMILES remaining (should be 0)
PASS: No data leakage


## Summary

In [5]:
print("=== Dataset Files ===")
print(f"  Original:  /home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings.csv")
print(f"             {len(bbbc_full):,} compounds, 672 dims")
print(f"  Filtered:  /home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings_filtered.csv")
print(f"             {len(bbbc_filtered):,} compounds, 672 dims")
print(f"  Removed:   {len(bbbc_full) - len(bbbc_filtered):,} ADMET test SMILES")
print(f"\n=== Hydra configs ===")
print(f"  Original:  tasks=bbbc047  (uses bbbc047_smiles_embeddings.csv)")
print(f"  Filtered:  tasks=bbbc047_filtered  (needs new config pointing to _filtered.csv)")

=== Dataset Files ===
  Original:  /home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings.csv
             109,645 compounds, 672 dims
  Filtered:  /home/shpark/prj-molrepr/data/bbbc047/bbbc047_smiles_embeddings_filtered.csv
             108,671 compounds, 672 dims
  Removed:   974 ADMET test SMILES

=== Hydra configs ===
  Original:  tasks=bbbc047  (uses bbbc047_smiles_embeddings.csv)
  Filtered:  tasks=bbbc047_filtered  (needs new config pointing to _filtered.csv)
